# Kết quả thu thập và chất lượng dữ liệu

Cập nhật ngày 06/11/2026, sử dụng audit các ngày 02–04/11. Nhóm báo cáo đã chạy hết 5.000 ID/người. Các tệp audit đã nộp xác nhận đủ record trên bốn dải TV2–TV5; TV1 chưa có tệp đối chiếu trong nguồn được cung cấp.

## Tiến độ xử lý đầu vào

TV3 và TV4 đã bổ sung kết quả audit đúng dải 5.000 ID. Cả hai có 0 ID chưa ghi nhận và 0 lookup chưa giải quyết. TV5 có đủ record nhưng còn bốn lookup lỗi. Một record có thể là lần thử thất bại, nên tiến độ được đọc cùng trạng thái lookup.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '_config.yml').exists())
sys.path.insert(0, str(ROOT / 'utils'))
from preliminary_results import (load_members, progress_rows, assignment_outcomes,
    quality_rows, error_groups, performance_rows, run_range, session_members, coverage_rows, cumulative_rows)
members = load_members(ROOT)
sessions = session_members(ROOT, members)
def table(rows):
    frame = pd.DataFrame(rows, dtype=object)
    frame = frame.where(frame.notna(), 'Chưa đủ bằng chứng')
    def number(value):
        if isinstance(value, float):
            return f'{value:,.2f}'.replace(',', '_').replace('.', ',').replace('_', '.')
        if isinstance(value, int) and not isinstance(value, bool):
            return f'{value:,}'.replace(',', '.')
        return str(value)
    display(HTML(frame.map(number).to_html(index=False, escape=True, border=0, classes='dataframe')))


In [ ]:
table(progress_rows(members))

Bốn dải có audit tương ứng 20.000 ID được ghi nhận, trong đó 19.996 lookup thành công và bốn còn lỗi. Đây là kết quả của các dải đã đối chiếu, chưa phải tổng kết 25.000 ID. Cờ hoàn tất audit còn xét validation và không được dùng thay riêng số ID đã xử lý.

## Dữ liệu được ghi nhận trên toàn dải

Số version dưới đây được cộng từ dữ liệu mà validator đọc được. Nếu một kết quả kiểm định không đọc được, số version có thể thiếu; chưa dùng cột này thay tổng version xác định từ toàn bộ Discovery.

In [ ]:
table(assignment_outcomes(members))

### Source, kích thước corpus và references

Các ZIP mới của TV3/TV4 đã có audit, nhưng chưa có thống kê tích lũy đầy đủ. Statistics TV3 vẫn chỉ chứa 828 ID; bản TV4 cuối và TV5 trước đó chỉ chứa sáu ID. Hai phiên TV4 có audit bổ sung chưa có statistics của snapshot hoàn tất.

Các chỉ số sau cần được xuất từ checkpoint và dữ liệu tích lũy trên đúng dải 5.000 ID. Chúng sẽ được bổ sung khi có tệp thống kê toàn dải; những giá trị của lượt resume cuối không được dùng làm trung bình toàn corpus.

In [ ]:
available = [m for m in members if m['cumulative']]
if available:
    table(cumulative_rows(members))
    table([{'Thành viên': m['member'], 'Paper có cặp đo': m['cumulative']['sizes']['paired_papers'],
            'Trước lọc (bytes/paper)': m['cumulative']['sizes']['mean_before_bytes'],
            'Sau lọc (bytes/paper)': m['cumulative']['sizes']['mean_after_bytes'],
            'References/paper truy vấn được': m['cumulative']['references']['mean_per_retrieved_paper']}
           for m in available])
    table([{'Thành viên':m['member'],
            'Paper truy vấn references đọc được':m['cumulative']['counts']['retrieved_reference_papers'],
            'Paper Discovery thành công':m['cumulative']['counts']['discovery_success_papers'],
            'References đủ metadata':m['cumulative']['references']['complete_numerator'],
            'References đã đọc':m['cumulative']['references']['complete_denominator'],
            'Diagnostics':len(m['cumulative']['diagnostics'])} for m in available])
else:
    table([{'Chỉ số cần bổ sung': 'Source/corpus thành công và paper đủ mọi version', 'Nguồn cần đọc': 'Discovery, checkpoint download/extract và tệp thực tế'},
           {'Chỉ số cần bổ sung': 'Kích thước trung bình trước và sau lọc', 'Nguồn cần đọc': 'Cùng tập paper/version có đủ sidecar trích xuất'},
           {'Chỉ số cần bổ sung': 'References, trung bình và tỷ lệ metadata đầy đủ', 'Nguồn cần đọc': 'Checkpoint và tệp references hiện hành từng paper'}])

Kích thước trước/sau lọc cần cặp đo trên cùng tập paper. References rỗng hợp lệ được tính vào mẫu số trung bình; lỗi API và chưa được index được báo riêng. Công cụ xuất thống kê tích lũy được liên kết ở [phụ lục](audit-appendix.ipynb), không yêu cầu crawl lại 5.000 ID.

## Chất lượng dữ liệu

TV3 có 4.820 paper đạt và 180 chưa đạt; TV4 có 4.882 đạt và 118 chưa đạt. Tỷ lệ dưới đây dùng số paper đã được kiểm định làm mẫu số, không gồm lookup chưa phát hiện được paper. Kết quả phản ánh ngày audit được nộp.

In [ ]:
quality = quality_rows(members)
table(quality)
verified = [r for r in quality if r['Paper đã kiểm định'] is not None]
fig, ax = plt.subplots(figsize=(8,4))
ax.bar([r['Thành viên'] for r in verified], [r['Đạt'] for r in verified], label='Đạt kiểm định')
ax.bar([r['Thành viên'] for r in verified], [r['Còn lỗi'] for r in verified],
       bottom=[r['Đạt'] for r in verified],label='Còn lỗi')
ax.set_ylabel('Paper đã kiểm định');ax.legend();plt.tight_layout();plt.show()

Các paper chưa đạt có thể vướng một version, một stage hoặc quy tắc kiểm tra. Cần xem nguyên nhân cụ thể trước khi quyết định tải lại. Audit TV2 có cờ hoàn tất mâu thuẫn với số fail; số đếm gốc được giữ ở phụ lục để xác minh tiếp.

### Nhóm vấn đề được validator ghi nhận

Mỗi paper chỉ đếm một lần trong từng nhóm; một paper có thể thuộc nhiều nhóm. Các cột không cộng thành tổng paper fail. Đây là phân loại thông báo kiểm định, chưa khẳng định nguyên nhân gốc hoặc trách nhiệm người chạy.

In [ ]:
table(error_groups(members))

## Phạm vi kết luận

Bằng chứng hiện có xác nhận đủ record đầu vào ở TV2–TV5, đồng thời cho biết các lỗi lookup và validation còn mở trong những snapshot này. Chưa có audit TV1 và thống kê tích lũy đầy đủ của năm thành viên. Vì vậy chưa công bố số source hợp lệ, trung bình kích thước/references hoặc tỷ lệ thành công cuối cùng cho toàn bộ nhóm.